# fea-surrogate — SFEM training on Colab

Trains a per-node von Mises surrogate on [`cmudrc/SFEM`](https://huggingface.co/datasets/cmudrc/SFEM)
and exports weights plus a **conformal calibration record**, which is what
`SurrogateScreening/v1` refuses documents without.

Runtime → Change runtime type → **T4 GPU** before running.

### What this downloads

Only `Pytorch_Geometric_Files/processed/` — **11.9 GB** (`train_data.pt` 10.07 GB,
`val_data.pt` 1.79 GB). The dataset card says ~7 GB for the first; the actual LFS
size is 10.07 GB.

Skipped: `raw_*.tar.gz` (3.78 GB) and `Step_Files/` (~3 GB). The processed files
*are* the output of the authors' `process()` over those raw HDF5 files, so
fetching raw would mean hours of CPU to rebuild what we already download.

**What that costs.** Their `load_physics_data` commented out `StressTensor`,
`Facets` and `u`, so the processed set carries **von Mises only**. Displacement
is not in it. A deformation field means taking the raw tars and re-running
`process()` with `u` enabled.

### Two things to watch, because they decide whether the output is usable

1. **Units.** The dataset's geometry sits in a `[0,3]³` box and `y` is
   log1p-then-Gaussian normalised. `y_original` holds the physical von Mises.
   Cell 4 prints its magnitude so we can establish whether it is Pa or MPa
   before anything claims a unit.
2. **The normaliser is per-graph.** The authors' loader fits a
   `UnitGaussianNormalizer` on *each graph's own* `y`. That is fine for their
   comparative study, but at inference on a new part you do not have that
   graph's statistics. We fit a **global** normaliser on the training split
   instead, which is what makes an absolute prediction possible at all.


In [ ]:
import subprocess, sys, torch, os
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torch-geometric", "huggingface_hub"], check=True)
print("installed")

## 1. Download the PyG-ready split

In [ ]:
from huggingface_hub import snapshot_download
import os, time

t0 = time.time()
path = snapshot_download(
    repo_id="cmudrc/SFEM",
    repo_type="dataset",
    allow_patterns=["Pytorch_Geometric_Files/processed/*"],
    local_dir="/content/sfem",
    max_workers=8,
)
print(f"downloaded in {(time.time()-t0)/60:.1f} min -> {path}")
for root, _, files in os.walk("/content/sfem/Pytorch_Geometric_Files/processed"):
    for f in sorted(files):
        p = os.path.join(root, f)
        print(f"{os.path.getsize(p)/1e9:6.2f} GB  {f}")

## 2. Load, and inspect what is actually there

Nothing below is assumed. This cell prints the real attribute names, shapes and
magnitudes, and the rest of the notebook is written against what it reports.

In [ ]:
import torch
from torch_geometric.data import InMemoryDataset
from torch_geometric.transforms import Compose, Cartesian, PointPairFeatures

ROOT = "/content/sfem/Pytorch_Geometric_Files"

class SFEM(InMemoryDataset):
    """Read the authors' collated file without re-running their process()."""
    def __init__(self, root, split, transform=None):
        self.split = split
        super().__init__(root, transform)
        self.data, self.slices = torch.load(
            self.processed_paths[0], weights_only=False
        )

    @property
    def raw_file_names(self):
        return []

    @property
    def processed_file_names(self):
        return [f"{self.split}_data.pt"]

    def download(self):
        pass

    def process(self):
        pass

transform = Compose([Cartesian(), PointPairFeatures()])
train_ds = SFEM(ROOT, "train", transform=transform)
val_ds = SFEM(ROOT, "val", transform=transform)
print(f"train graphs: {len(train_ds)}   val graphs: {len(val_ds)}")

g = train_ds[0]
print("\n--- attributes on one graph ---")
for key in sorted(g.keys()):
    v = g[key]
    if torch.is_tensor(v):
        print(f"  {key:12s} {tuple(v.shape)}  {v.dtype}")
    else:
        print(f"  {key:12s} {type(v).__name__}  {str(v)[:60]}")

print("\n--- target magnitudes, which decide the unit ---")
if hasattr(g, "y_original"):
    ys = torch.cat([train_ds[i].y_original.reshape(-1) for i in range(200)])
    print(f"  y_original  min {ys.min():.4g}  median {ys.median():.4g}  max {ys.max():.4g}")
    print("  -> E = 2.303 GPa; judge Pa vs MPa from these numbers")
else:
    print("  no y_original present; inverse transform must be derived from y")
print(f"  y (normalised)  min {g.y.min():.3f}  max {g.y.max():.3f}")
print(f"\nnode feature width: x = {g.x.shape[1]} columns")

## 3. Global normaliser and the peak-stress target

We predict the **per-node field**, and take its maximum as `von_mises_peak` —
the scalar the `SurrogateScreening/v1` crosscheck compares against the
closed-form `mechanics` answer.

In [ ]:
import numpy as np

def peak_original(ds, n=None):
    n = len(ds) if n is None else min(n, len(ds))
    return torch.tensor([ds[i].y_original.max().item() for i in range(n)])

# Global log-space statistics from the training split only.
sample = torch.cat([
    torch.log1p(train_ds[i].y_original.reshape(-1))
    for i in range(min(1000, len(train_ds)))
])
G_MEAN, G_STD = sample.mean().item(), sample.std().item()
print(f"global log1p normaliser: mean={G_MEAN:.4f} std={G_STD:.4f}")

def encode(y_phys):
    return (torch.log1p(y_phys) - G_MEAN) / (G_STD + 1e-5)

def decode(y_norm):
    return torch.expm1(y_norm * (G_STD + 1e-5) + G_MEAN)

tr_peak = peak_original(train_ds, 500)
print(f"train peak von Mises: median {tr_peak.median():.4g}  max {tr_peak.max():.4g}")

## 4. Baseline first

A plain geometric-feature regressor on the peak. This is the number the GNN has
to beat — if it does not, that is worth knowing in an hour rather than a week.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

def geom_features(d):
    pos = d.pos
    ext = (pos.max(0).values - pos.min(0).values)
    load = d.x[0, -3:]                      # one-hot load class
    return torch.cat([
        ext,
        ext.prod().reshape(1),
        torch.tensor([float(pos.shape[0])]),
        load,
    ]).numpy()

def build_xy(ds, n):
    n = min(n, len(ds))
    X = np.stack([geom_features(ds[i]) for i in range(n)])
    y = np.log1p(np.array([ds[i].y_original.max().item() for i in range(n)]))
    return X, y

Xtr, ytr = build_xy(train_ds, 3000)
Xva, yva = build_xy(val_ds, 800)

base = GradientBoostingRegressor(random_state=0).fit(Xtr, ytr)
pred = base.predict(Xva)
print(f"BASELINE  log-space MAE {mean_absolute_error(yva, pred):.4f}")
print(f"          physical MAE {mean_absolute_error(np.expm1(yva), np.expm1(pred)):.4g}")

## 5. Mesh GNN for the per-node field

In [ ]:
import torch.nn as nn
from torch_geometric.loader import DataLoader
from torch_geometric.nn import SAGEConv

dev = "cuda" if torch.cuda.is_available() else "cpu"

class StressGNN(nn.Module):
    def __init__(self, in_dim, hidden=128, layers=5):
        super().__init__()
        self.inp = nn.Linear(in_dim, hidden)
        self.convs = nn.ModuleList(
            [SAGEConv(hidden, hidden) for _ in range(layers)]
        )
        self.norms = nn.ModuleList(
            [nn.LayerNorm(hidden) for _ in range(layers)]
        )
        self.out = nn.Sequential(
            nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden, 1)
        )

    def forward(self, d):
        h = self.inp(d.x)
        for conv, norm in zip(self.convs, self.norms):
            h = h + torch.nn.functional.silu(norm(conv(h, d.edge_index)))
        return self.out(h)

model = StressGNN(train_ds[0].x.shape[1]).to(dev)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=8)

EPOCHS = 8   # raise once the loop is known to work
for epoch in range(EPOCHS):
    model.train(); total = 0.0; seen = 0
    for batch in train_loader:
        batch = batch.to(dev)
        target = encode(batch.y_original.reshape(-1, 1).to(dev))
        opt.zero_grad()
        loss = torch.nn.functional.smooth_l1_loss(model(batch), target)
        loss.backward(); opt.step()
        total += loss.item() * batch.num_graphs; seen += batch.num_graphs
    print(f"epoch {epoch+1}/{EPOCHS}  train {total/seen:.4f}")

## 6. Split-conformal calibration on the peak

This is the part that makes the estimate admissible. The band is derived from
held-out residuals with the finite-sample correction, so its coverage is a
measured property rather than an assertion.

In [ ]:
model.eval()
res, truths = [], []
with torch.no_grad():
    for batch in val_loader:
        batch = batch.to(dev)
        node_pred = decode(model(batch).reshape(-1)).cpu()
        # peak per graph
        idx = batch.batch.cpu()
        for gi in range(batch.num_graphs):
            p = node_pred[idx == gi].max().item()
            t = batch.y_original.reshape(-1).cpu()[idx == gi].max().item()
            res.append(abs(p - t)); truths.append(t)

res = np.array(res)
ALPHA = 0.10                      # 90% target coverage
n = len(res)
q_level = min(1.0, np.ceil((n + 1) * (1 - ALPHA)) / n)
Q = float(np.quantile(res, q_level, method="higher"))
covered = float((res <= Q).mean())
print(f"calibration set n={n}")
print(f"half-width Q = {Q:.4g}   empirical coverage = {covered:.3f} (target {1-ALPHA})")

## 7. Export weights and the calibration record

`calibration.set_sha256` must identify the exact calibration set, because the
schema refuses a coverage figure that cannot be attributed.

In [ ]:
import hashlib, json, datetime

torch.save(model.state_dict(), "/content/stress_gnn.pt")
weights_sha = hashlib.sha256(open("/content/stress_gnn.pt","rb").read()).hexdigest()
calib_sha = hashlib.sha256(res.tobytes()).hexdigest()

record = {
    "model": {"name": "sfem-mesh-gnn", "version": "0.1.0", "sha256": weights_sha},
    "coverage": 1 - ALPHA,
    "calibration": {
        "method": "split_conformal_absolute_residual",
        "set_size": int(n),
        "set_sha256": calib_sha,
    },
    "half_width": Q,
    "normaliser": {"log1p_mean": G_MEAN, "log1p_std": G_STD},
    "empirical_coverage": covered,
    "trained_at": datetime.datetime.now(datetime.UTC).isoformat(),
    "domain": {
        "material_class": "linear_elastic",
        "load_case": ["point"],
        "source_dataset": "cmudrc/SFEM",
    },
    "limitations": [
        "one material only (E = 2.303 GPa, nu = 0.4002)",
        "point loads only, at 200 N / 2000 N / 20000 N",
        "synthetic BrepGen geometry; no gear pairs or shafts",
        "screening estimate, not a strength certification",
    ],
}
with open("/content/calibration.json", "w") as f:
    json.dump(record, f, indent=2)
print(json.dumps(record, indent=2))

from google.colab import files
files.download("/content/stress_gnn.pt")
files.download("/content/calibration.json")

## What comes back

`stress_gnn.pt` and `calibration.json` — megabytes, not gigabytes. The dataset
stays here; `fea-surrogate` serves inference locally and needs none of it.

**Before this can emit a real `SurrogateScreening/v1`**, two things from cell 2
have to be settled: the physical unit of `y_original`, and how a new part's
geometry is mapped into the `[0,3]³` box the model was fitted in. Get those
wrong and the interval is precise nonsense — which is exactly what the domain
gate exists to refuse.
